# Exportação da Gold para CSV

**Objetivo:** exportar as tabelas da Gold de interesse dos usuários do portal
para CSV (um arquivo por tabela), conforme o case, para carga no banco do backend.

**Regras:**
- Último passo do Job, depois de todas as Gold.
- Lê só `workspace.gold`; não altera nenhuma tabela.
- Saída: `/Volumes/workspace/gold/exportacao/<tabela>.csv`
  (UTF-8, cabeçalho, separador vírgula, decimal com ponto, datas ISO, nulo = vazio,
  arrays juntados com `|`).
- Os CSVs NUNCA vão para o GitHub (repo público; ver `.gitignore`).
- Limpar outputs antes do push.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, MapType, StructType

CAT, SCHEMA = "workspace", "gold"

tabelas = [r.tableName for r in spark.sql(f"SHOW TABLES IN {CAT}.{SCHEMA}").collect()
           if not r.isTemporary]

inventario = []
for t in sorted(tabelas):
    df = spark.table(f"{CAT}.{SCHEMA}.{t}")
    complexas = [f.name for f in df.schema.fields
                 if isinstance(f.dataType, (ArrayType, MapType, StructType))]
    inventario.append((t, df.count(), len(df.columns), ", ".join(complexas) or "-"))

display(spark.createDataFrame(inventario,
        "tabela string, linhas long, colunas int, colunas_complexas string"))

In [0]:
TABELAS_EXPORT = [
    "dim_mes", "dim_uc", "dim_alimentador", "fato_uc_mes",
    "uc_consumo_comparativo", "uc_interrupcoes", "uc_atendimentos",
    "agg_receita_mensal", "agg_preco_volume", "agg_kpis_mensal",
    "agg_dec_fec_mensal", "fato_alimentador_mes",
    "tb_fila_perdas", "tb_fila_perdas_auditoria",
]

# Garantia 1: toda tabela da lista existe e tem linhas
faltando = [t for t in TABELAS_EXPORT if t not in tabelas]
assert not faltando, f"Tabelas da lista não existem na Gold: {faltando}"
vazias = [t for t in TABELAS_EXPORT if spark.table(f"{CAT}.{SCHEMA}.{t}").limit(1).count() == 0]
assert not vazias, f"Tabelas vazias: {vazias}"

# Garantia 2: nenhum CPF/CNPJ sem máscara em NENHUMA tabela exportada
MASCARA = r"^(\*\*\*\.\d{3}\.\d{3}-\*\*|\*\*\.\d{3}\.\d{3}/\d{4}-\*\*)$"
for t in TABELAS_EXPORT:
    df = spark.table(f"{CAT}.{SCHEMA}.{t}")
    for c in [c for c in df.columns if "cpf" in c.lower() or "cnpj" in c.lower()]:
        fora = df.where(F.col(c).isNotNull() & ~F.col(c).rlike(MASCARA)).count()
        print(f"{t}.{c}: {fora} fora da máscara")
        assert fora == 0, f"{t}.{c} tem {fora} documentos sem máscara"

# Informação: versão da fila que será exportada
display(spark.table(f"{CAT}.{SCHEMA}.tb_fila_perdas")
        .select("versao_regra", "mes_referencia", "id_execucao").distinct())

print(f"OK: {len(TABELAS_EXPORT)} tabelas prontas para exportar")

In [0]:
import re
from pyspark.sql.types import ArrayType

VOLUME = "exportacao"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CAT}.{SCHEMA}.{VOLUME}")
DESTINO = f"/Volumes/{CAT}/{SCHEMA}/{VOLUME}"

OPCOES_CSV = {"header": "true", "encoding": "UTF-8", "sep": ",",
              "quote": '"', "escape": '"'}

def preparar_para_csv(df):
    """Arrays viram texto separado por '|' (CSV não suporta tipos complexos)."""
    for f in df.schema.fields:
        if isinstance(f.dataType, ArrayType):
            df = df.withColumn(f.name, F.array_join(F.col(f.name).cast("array<string>"), "|"))
    return df

def exportar_tabela(tabela):
    df = preparar_para_csv(spark.table(f"{CAT}.{SCHEMA}.{tabela}"))
    tmp, final = f"{DESTINO}/_tmp_{tabela}", f"{DESTINO}/{tabela}.csv"

    # Spark grava uma pasta; com coalesce(1) ela tem um único part-*.csv
    df.coalesce(1).write.mode("overwrite").options(**OPCOES_CSV).csv(tmp)
    parte = [f.path for f in dbutils.fs.ls(tmp) if re.match(r"part-.*\.csv$", f.name)]
    assert len(parte) == 1, f"{tabela}: esperado 1 part-*.csv, achei {len(parte)}"

    dbutils.fs.rm(final)                      # remove a versão anterior, se houver
    dbutils.fs.cp(parte[0], final)
    dbutils.fs.rm(tmp, recurse=True)

    # Reconciliação: o CSV relido tem as mesmas linhas da tabela?
    n_tab = spark.table(f"{CAT}.{SCHEMA}.{tabela}").count()
    n_csv = spark.read.options(**OPCOES_CSV, multiLine="true").csv(final).count()
    assert n_tab == n_csv, f"{tabela}: tabela {n_tab} x CSV {n_csv}"
    return (tabela, n_tab, n_csv)

# Teste com duas tabelas
for t in ["dim_mes", "tb_fila_perdas"]:
    print(exportar_tabela(t))

# Conferência visual: primeiras linhas do CSV da fila, como texto bruto
print(dbutils.fs.head(f"{DESTINO}/tb_fila_perdas.csv", 1500))

In [0]:
import time

resultado = []
for t in TABELAS_EXPORT:
    ini = time.time()
    tabela, n_tab, n_csv = exportar_tabela(t)
    resultado.append((tabela, n_tab, n_csv, round(time.time() - ini, 1)))
    print(f"OK {tabela}: {n_tab} linhas")

display(spark.createDataFrame(resultado,
        "tabela string, linhas_tabela long, linhas_csv long, segundos double"))

# Arquivos finais no volume (nenhuma pasta _tmp_ deve sobrar)
arquivos = [(f.name, round(f.size / 1024**2, 1)) for f in dbutils.fs.ls(DESTINO)]
display(spark.createDataFrame(arquivos, "arquivo string, tamanho_mb double"))
print("Total:", round(sum(a[1] for a in arquivos), 1), "MB")

In [0]:
import csv, hashlib, os
from datetime import datetime, timezone

def sha256_arquivo(caminho, bloco=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for parte in iter(lambda: f.read(bloco), b""):
            h.update(parte)
    return h.hexdigest()

fila = spark.table(f"{CAT}.{SCHEMA}.tb_fila_perdas").select("versao_regra", "id_execucao").first()
exportado_em = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
linhas_por_tabela = {r[0]: r[1] for r in resultado}   # contagens já reconciliadas na célula 4

manifesto = []
for t in TABELAS_EXPORT:
    caminho = f"{DESTINO}/{t}.csv"
    colunas = spark.table(f"{CAT}.{SCHEMA}.{t}").columns
    manifesto.append({
        "tabela": t,
        "arquivo": f"{t}.csv",
        "linhas": linhas_por_tabela[t],
        "n_colunas": len(colunas),
        "colunas": "|".join(colunas),
        "tamanho_bytes": os.path.getsize(caminho),
        "sha256": sha256_arquivo(caminho),
        "versao_regra_fila": fila["versao_regra"],
        "id_execucao_fila": fila["id_execucao"],
        "exportado_em_utc": exportado_em,
    })

with open(f"{DESTINO}/_manifesto.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(manifesto[0].keys()))
    w.writeheader()
    w.writerows(manifesto)

assert len(manifesto) == len(TABELAS_EXPORT)
display(spark.createDataFrame(manifesto).select("tabela", "linhas", "n_colunas", "tamanho_bytes", "sha256"))
print(f"Manifesto gravado em {DESTINO}/_manifesto.csv ({exportado_em})")